# **Algorithmic Recourse: Practice**

**Hi, friends!**

Tamara walked through every lesson of the module "Deep Dive into Counterfactuals": she is 63, a former accountant, and she applied for an ML engineer position. The screening model invites a candidate to an interview when $p_{\text{invite}} \ge 0.5$, and it gave Tamara $p = 0.03$. Her application features are: age 63, 2 completed courses, 3 portfolio projects, 0.5 years of commercial experience. Age cannot be changed, and the model barely looks at it; courses, projects and experience are things Tamara can add.

In the lessons each method answered Tamara on its own. Here we gather the answers in one place and compare them on the properties from the lesson "Metrics and the main empirical fact": validity, proximity, sparsity, plausibility, actionability, robustness.

In this practice you will:
- rebuild the data and the random-forest model from the lessons and check that Tamara gets $p = 0.03$;
- find the minimal change in the Wachter sense with a grid search and see how the weight $\lambda$ decides whether the counterfactual becomes valid;
- get three different plans from DiCE with age frozen and measure their proximity, sparsity and diversity;
- add a pull towards the prototype of the "invited" class and see the counterfactual become more typical and move further from Tamara;
- build FACE yourself (a k-NN graph, KDE density, Dijkstra) and compare a step-by-step path with the Wachter jump;
- set action costs in the spirit of Ustun, find the cheapest plan and a case where no recourse exists;
- check every plan you found on 15 retrained models and under execution noise, as ROAR and PRR do;
- put everything into one table of properties.

The notebook `PRACTICE_counterfactual_explanations` works with DiCE on another task (house price regression); this notebook is about the properties of recourse for one person.

**Happy coding!**

### Which libraries we use and why not the others

The module lessons name four tabular libraries. On Colab (Python 3.13, numpy 2.1.3) only one of them works, so we write the other methods ourselves in a few lines, following the formulas from the lessons.

| Library | What it offers | On Colab |
|---|---|---|
| `dice-ml` 0.12 | DiCE: genetic, random, kdtree | installs, we use it |
| `alibi` 0.9.6 | Wachter, `CounterfactualProto` | installing it downgrades numpy 2.1.3 → 1.26.4, Pillow → 10.4.0, transformers 5.18.0 → 4.57.6 and blis → 0.7.11 and breaks the rest of the environment; we write Wachter and prototypes ourselves as a grid search |
| CARLA (`carla-recourse` 0.0.5) | Growing Spheres, REVISE, C-CHVAE, FACE, ROAR | does not install: it needs tensorflow 1.14, which has no builds for Python 3.13 (plus numpy 1.19 and torch 1.7); we write FACE and the robustness check ourselves |
| OmniXAI 1.3.1 | a wrapper over DiCE and other explainers | installing it downgrades numpy to 1.23.5 and scikit-learn to 1.1.3 |
| `actionable-recourse` 1.0.1 | Ustun: integer programming | installs together with `cplex` and `mip==1.12`, but fails on the first `Flipset` call: it uses `np.float_`, which numpy 2 removed; besides, the method is built for a linear model, and Tamara's model is a forest. We compute action costs by enumerating every integer action |

All of this was checked by installing the packages into an environment with Colab's package versions as of 10.10.2026.

In [ ]:
!pip install dice-ml==0.12 -q

In [ ]:
import random, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
import dice_ml

pd.set_option('display.precision', 3)

## 1. Data and model from the lessons

The code repeats the step "Growing Spheres for Tamara" of lesson 2: the same 2500 synthetic applications with seed 0, the same score (age does not enter it), the same model — standardization plus a random forest of 200 trees. We only wrapped the generation into a function `make_data(seed)`: section 7 needs it to retrain the model on new data, like the function `make(seed)` from the step "Robust recourse for Tamara" of lesson 5. With `seed=0` you get exactly the data from the lessons.

In [ ]:
cont = ["age", "courses", "projects", "exp"]
skill = ["courses", "projects", "exp"]           # what Tamara can change

def make_data(seed, n=2500):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "age":      rng.normal(42, 12, n).clip(22, 70),
        "courses":  rng.normal(4, 2, n).clip(0, 15),
        "projects": rng.normal(3, 2, n).clip(0, 20),
        "exp":      rng.normal(3, 2.5, n).clip(0, 20)})
    score = 0.5*df.courses + 0.7*df.projects + 0.6*df.exp + rng.normal(0, 1, n)  # age does NOT enter
    df["invited"] = (score > np.median(score)).astype(int)
    return df

def make_model(df):
    return Pipeline([
        ("sc", ColumnTransformer([("s", StandardScaler(), cont)])),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=0))
    ]).fit(df[cont], df["invited"])

df = make_data(0)
model = make_model(df)

tamara = np.array([63., 2., 3., 0.5])              # age, courses, projects, experience
def p(x, m=model): return m.predict_proba(pd.DataFrame([x], columns=cont))[0, 1]
def p_many(X, m=model): return m.predict_proba(pd.DataFrame(X, columns=cont))[:, 1]

print("p(Tamara is invited) =", round(p(tamara), 3))
print("share of invited in the data:", df.invited.mean())
df.head()

Let us check the promise "the model does not look at age". Age is not in the score, but the forest can still split on it by fitting noise.

In [ ]:
imp = model.named_steps["clf"].feature_importances_
print("forest feature importances:", dict(zip(cont, imp.round(3).tolist())))
print("p at another age with the same skills:",
      {a: round(float(p([a, 2., 3., 0.5])), 3) for a in (25, 40, 63, 70)})

The forest gave age about 9% of the importance, yet Tamara's $p$ barely depends on age: from 0.025 to 0.05 across the whole range. So the answer "get younger" would not help Tamara even if it were possible — and every method below keeps age frozen at 63.

### How we measure the properties

We describe every counterfactual $x'$ we find with the same numbers (their definitions are in the lesson "Metrics and the main empirical fact"):

- **validity** — $p(x') \ge 0.5$ under the current model;
- **proximity** — the $L_1$ distance over skills, each skill divided by its MAD (median absolute deviation), as in Wachter; smaller means closer to Tamara;
- **sparsity** — how many of the three skills changed;
- **plausibility** — the mean distance to the 5 nearest real applications in standardized skills; smaller means more typical (for Tamara herself it is 0.177);
- **actionability** — age untouched and no skill decreased; for such plans we compute the cost in months of effort (section 6);
- **robustness** — the share of retrained models and the share of noisy executions under which the plan stays valid (section 7).

In [ ]:
mad = (df[skill] - df[skill].median()).abs().median().values
sd = df[skill].std().values
knn = NearestNeighbors(n_neighbors=5).fit(df[skill].values / sd)
unit_cost = np.array([1., 2., 12.])             # months per +1 course, +1 project, +1 year of experience (section 6)

def proximity(x):    return np.abs((x[1:] - tamara[1:]) / mad).sum()
def sparsity(x):     return int((np.abs(x[1:] - tamara[1:]) > 1e-9).sum())
def plausibility(x): return knn.kneighbors([x[1:] / sd])[0].mean()
def actionable(x):   return bool(x[0] == tamara[0] and (x[1:] >= tamara[1:] - 1e-9).all())
def cost(x):         return float((x[1:] - tamara[1:]) @ unit_cost)

def describe(x):
    return {"courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
            "p": round(p(x), 3), "valid": p(x) >= 0.5, "proximity": round(proximity(x), 2),
            "sparsity": sparsity(x), "plausibility": round(plausibility(x), 3),
            "actionable": actionable(x)}

found = {}                                       # counterfactuals of all methods go here
pd.DataFrame([describe(tamara)], index=["Tamara"])

## 2. Wachter: the minimal change

The basic formulation from the step "What do we already know about counterfactuals?" of lesson 1:

$$\arg\min_{x'}\ \max_{\lambda}\ \lambda\big(f(x') - y'\big)^2 + d(x, x'),$$

where $d$ is our proximity ($L_1$ scaled by MAD) and $y' = 1$. Alibi solves this with gradient descent (for a black box the gradient is estimated numerically), and the same search underlies `CounterfactualProto`. A forest has no gradient and there are only three skills, so we replace the descent with an exhaustive grid search with step 0.1: courses from 0 to 8, projects from 0 to 9, experience from 0 to 8 years, age 63. That is about 600 thousand points, and the forest scores them in a few seconds.

In [ ]:
t0 = time.time()
c, pr, e = np.meshgrid(np.arange(0, 8.01, 0.1), np.arange(0, 9.01, 0.1), np.arange(0, 8.01, 0.1), indexing="ij")
G = np.column_stack([np.full(c.size, 63.), c.ravel(), pr.ravel(), e.ravel()])
PG = p_many(G)
DG = (np.abs(G[:, 1:] - tamara[1:]) / mad).sum(1)       # proximity of every grid point
print(len(G), "points, %.1f s" % (time.time() - t0))

First let us see what the weight $\lambda$ does: for a fixed $\lambda$ we minimize $\lambda(p(x') - 1)^2 + d(x, x')$ over the whole grid.

In [ ]:
rows = []
for lam in [1, 10, 100, 1000]:
    k = np.argmin(lam * (PG - 1)**2 + DG)
    rows.append({"lambda": lam, **describe(G[k])})
pd.DataFrame(rows).set_index("lambda")

At $\lambda = 1$ the prediction penalty is cheaper than any step, and the optimum is Tamara herself with $p = 0.03$: the counterfactual is invalid. As $\lambda$ grows the point crosses the boundary, and at large $\lambda$ it goes deeper into the invited region, further from Tamara. That is why the formula has $\max_\lambda$: the weight is increased until the prediction becomes the desired one. In the limit the task is "the closest point among the valid ones".

**Task 1.** Among the grid points with $p \ge 0.5$, find the point with the smallest proximity `DG` and store it in `x_wachter`.

*Self-check question:* which skills does the Wachter counterfactual change, what is its $p$ and how many skills does it touch? Compare with Growing Spheres from the step "Growing Spheres for Tamara" of lesson 2 (courses $2\to2.2$, projects $3\to4.4$, experience $0.5\to1.6$).

<details><summary>Check yourself</summary>

The Wachter counterfactual: courses 2 (unchanged), projects $3 \to 4.4$, experience $0.5 \to 1.6$ years, $p = 0.505$, proximity 1.69; it changes 2 of the 3 skills. Growing Spheres in lesson 2 arrived at almost the same point (projects 4.4, experience 1.6) but also touched courses ($2 \to 2.2$): there the point was found by random sampling in a ball, while an exhaustive search with the $L_1$ distance leaves courses where they are. Solution: `x_wachter = G[np.argmin(np.where(PG >= 0.5, DG, np.inf))]`.
</details>

In [ ]:
x_wachter = ...  # Your code here
found["Wachter"] = x_wachter
pd.DataFrame([describe(x_wachter)], index=["Wachter"])

The Wachter counterfactual sits right on the boundary: $p$ is barely above 0.5. This is a property of the minimal solution, and in section 7 we will see what it costs.

## 3. DiCE: several different plans

First let us repeat the step "DiCE for Tamara" of lesson 2 as it is: genetic search, three counterfactuals, age excluded from `features_to_vary`, the query written with integers. In the lesson the search ran without a seed, so the third counterfactual changed from run to run; here we fix `random` and `numpy`, and all three plans reproduce. Next to the `invited` column printed by DiCE we put $p$ from the model itself.

In [ ]:
d = dice_ml.Data(dataframe=df, continuous_features=cont, outcome_name="invited")
m = dice_ml.Model(model=model, backend="sklearn")
query_int = pd.DataFrame([{"age": 63, "courses": 2, "projects": 3, "exp": 0.5}])   # as in the lesson

random.seed(0); np.random.seed(0)
cf_int = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query_int, total_CFs=3, desired_class=1, features_to_vary=skill)
out = cf_int.cf_examples_list[0].final_cfs_df.copy()
print(out.dtypes.to_dict())
out["model p"] = p_many(out[cont].values.astype(float)).round(3)
out

The first two plans match the lesson table: courses 3, projects 3, experience 3.2 and courses 2, projects 4, experience 1.6. DiCE prints `invited = 1` for them, yet the model gives $p = 0.285$ and $p = 0.255$: both plans are **invalid**. The cause is dtypes: the query stores courses and projects as integers, and DiCE casts the counterfactuals it found to the query dtypes, dropping the fractional part after the search. Inside the search there were, for example, 3.4 courses, the output kept 3, and the point fell back across the boundary. Hence the rule: the validity of a counterfactual is checked with the model's prediction, not with the column a library prints.

Let us write the query with floats and repeat the search.

In [ ]:
query = pd.DataFrame([{"age": 63., "courses": 2., "projects": 3., "exp": 0.5}])

random.seed(0); np.random.seed(0)
t0 = time.time()
cf_gen = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill)
print("genetic: %.2f s" % (time.time() - t0))
dice_gen = cf_gen.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_gen], index=["DiCE-1", "DiCE-2", "DiCE-3"])

Now all three plans are valid. The first two are the same paths as in the lesson, but without the dropped fractional part: DiCE-1 is 3.4 courses and 3.2 years of experience (the lesson kept 3 courses), DiCE-2 is 2.5 courses, 4.8 projects and 1.6 years of experience (the lesson kept 2 courses and 4 projects). The third plan is 5.5 courses with 2.9 projects: DiCE lowered the number of projects. `features_to_vary` froze age, but it does not restrict the direction of changes; such a plan is formally valid, yet it cannot be carried out (a portfolio project is not "undone"). DiCE has `permitted_range` for this.

For comparison, let us run the same query with the `random` method: it samples points in the permitted region and keeps the valid ones.

In [ ]:
random.seed(0); np.random.seed(0)
cf_rnd = dice_ml.Dice(d, m, method="random").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill, random_seed=0)
dice_rnd = cf_rnd.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_rnd], index=["random-1", "random-2", "random-3"])

**Task 2.** DiCE measures the diversity of a set with the determinant of a similarity matrix (the step "Formal statement of DiCE" of lesson 2). Let us compute a simpler quantity: the mean pairwise $L_1$ distance scaled by MAD between the plans of a set. Complete the function `diversity(X)` for an array of plans `X` (rows are plans, columns are `cont`).

*Self-check question:* which method, `genetic` or `random`, has the larger mean proximity of its plans and the larger diversity of its set? What are the numbers?

<details><summary>Check yourself</summary>

`genetic`: mean proximity 2.55, diversity 3.42; `random`: mean proximity 8.19, diversity 10.82. The `random` method is further from Tamara and more diverse: it samples the whole permitted region and does not pull the plans towards the object. Solution: the mean over pairs $i < j$ of `np.abs((X[i, 1:] - X[j, 1:]) / mad).sum()`.
</details>

In [ ]:
def diversity(X):
    # Your code here
    ...

for name, X in [("genetic", dice_gen), ("random", dice_rnd)]:
    print("%-8s mean proximity = %.2f   mean sparsity = %.2f   diversity = %.2f"
          % (name, np.mean([proximity(x) for x in X]), np.mean([sparsity(x) for x in X]), diversity(X)))

Now let us forbid decreasing skills: `permitted_range` sets a corridor that starts at Tamara's current value.

In [ ]:
random.seed(0); np.random.seed(0)
cf_pr = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill,
    permitted_range={"courses": [2, 15], "projects": [3, 20], "exp": [0.5, 20]})
dice_pr = cf_pr.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
print("diversity = %.2f" % diversity(dice_pr))
pd.DataFrame([describe(x) for x in dice_pr], index=["DiCE+range-1", "DiCE+range-2", "DiCE+range-3"])

With the corridor all three plans are actionable, but they change only experience, and the diversity of the set dropped from 3.42 to 0.47. The actionability constraint was paid for with diversity: this is the first conflict between properties in the notebook.

In [ ]:
found["DiCE-1"], found["DiCE-2"], found["DiCE-3"] = dice_gen
found["DiCE+range"] = dice_pr[0]

## 4. Prototypes: a counterfactual close to typical invited candidates

We build the prototype as in the step "Prototype and latent space for Tamara" of lesson 3: the mean of the 25 real invited applications closest to Tamara (distance in standardized features), with age kept at 63.

In [ ]:
sd_all = df[cont].std().values
inv = df[df.invited == 1][cont].values
proto = inv[np.argsort(np.linalg.norm((inv - tamara) / sd_all, axis=1))[:25]].mean(0)
proto[0] = 63                                   # age is immutable, keep it
pd.DataFrame([describe(proto)], index=["prototype"])

The numbers match the lesson (courses 3.3, projects 3.8, experience 2.2), and one thing deserves attention: the model does **not** invite the prototype itself, $p = 0.215$. The invited candidates closest to Tamara got their invitations partly thanks to the noise in the score, and their mean still lies before the boundary. So the prototype serves as a direction, not as an answer: it is used as a penalty inside the search, and the condition $p \ge 0.5$ takes care of validity.

Let us take the objective of the prototype method from the step "What is added to the objective" of lesson 3 without the autoencoder (the prototype itself partly plays its role):

$$L(x') = d(x, x') + \theta\,\Big\lVert \frac{x' - \text{proto}}{\text{sd}} \Big\rVert_2^2 \quad\text{subject to}\quad p(x') \ge 0.5.$$

**Task 3.** For each $\theta$ in the list, find the valid point of the grid from section 2 with the smallest $L$. The squared standardized distance to the prototype for every grid point is already computed in `DP`.

*Self-check question:* how do the proximity and the plausibility of the counterfactual change from $\theta = 0$ to $\theta = 10$?

<details><summary>Check yourself</summary>

At $\theta = 0$ (and at $\theta = 1$) the answer coincides with Wachter: proximity 1.69, plausibility 0.228. At $\theta = 10$: courses 3.5, projects 4.2, experience 2.5, proximity 3.16 (almost twice as large), plausibility 0.125 (smaller, that is more typical; Tamara herself has 0.177). Solution: `k = np.argmin(np.where(PG >= 0.5, DG + theta * DP, np.inf))`.
</details>

In [ ]:
DP = (((G[:, 1:] - proto[1:]) / sd)**2).sum(1)
rows, proto_cf = [], {}
for theta in [0, 1, 3, 10]:
    k = ...  # Your code here: index of the grid point
    proto_cf[theta] = G[k]
    rows.append({"theta": theta, **describe(G[k]), "to prototype": round(np.sqrt(DP[k]), 2)})
pd.DataFrame(rows).set_index("theta")

At small $\theta$ the prototype penalty is weaker than one grid step, and the answer coincides with Wachter. At $\theta = 10$ the counterfactual moves towards the prototype and lands in a denser region of applications, but its proximity grew from 1.69 to 3.16, while the plausibility distance dropped from 0.228 to 0.125. This is the trade-off of lesson 3: plausibility is paid for with proximity.

In [ ]:
found["Prototype θ=10"] = proto_cf[10]

## 5. FACE by hand: a path through dense regions

CARLA with FACE does not install on Colab, so we repeat the code of the step "Three requirements for Tamara's recourse" of lesson 4: the graph nodes are Tamara and the 2500 real applications (skills only), the edges connect 15 nearest neighbours in standardized skills, and the edge weight is $-\log \hat p\big(\tfrac{x_i + x_j}{2}\big)\cdot \lVert x_i - x_j \rVert$ with the KDE density at the edge midpoint. The target is an application that the model invites with $p \ge 0.6$ at Tamara's age and that itself lies in the denser half of the data. Dijkstra finds the path.

The only difference from the lesson: we compute the density at the edge midpoints with one KDE call for all edges at once instead of a loop. The numbers stay the same, and the cell runs several times faster.

In [ ]:
t0 = time.time()
X = df[skill].values.astype(float)
proba = model.predict_proba(df[cont].assign(age=63.))[:, 1]   # prediction for each application's skills at Tamara's age

pts = np.vstack([tamara[1:], X]); N = len(pts)  # Tamara is node 0
Z = (pts - X.mean(0)) / X.std(0)                # standardize for distances
kde = gaussian_kde(X.T)                         # data density

dist, idx = NearestNeighbors(n_neighbors=16).fit(Z).kneighbors(Z)
rows = np.repeat(np.arange(N), 15); cols = idx[:, 1:].ravel(); dd = dist[:, 1:].ravel()
dens = kde(((pts[rows] + pts[cols]) / 2).T)
w = np.maximum(-np.log(dens + 1e-12) * dd, 1e-6)  # -log(density) · distance
Gr = csr_matrix((w, (rows, cols)), shape=(N, N)); Gr = Gr.maximum(Gr.T)

dens_all = kde(X.T)
targets = np.where((proba >= 0.6) & (dens_all >= np.quantile(dens_all, 0.5)))[0] + 1
dpath, pred = dijkstra(Gr, indices=0, return_predecessors=True)
best = targets[np.argmin(dpath[targets])]

path = [best]
while path[-1] != 0: path.append(pred[path[-1]])
path = path[::-1]
print("%.1f s" % (time.time() - t0))
for k in path: print(np.round(pts[k], 2))
x_face = np.r_[63., pts[best]]
found["FACE"] = x_face
pd.DataFrame([describe(x_face)], index=["FACE"])

The path matches the lesson: Tamara $[2, 3, 0.5] \to [2, 3.55, 0.88] \to [2.18, 4, 1.51] \to [2.39, 4.37, 1.8]$, and the final application has $p = 0.625$ (the forest in `describe` gives the same number, since the endpoint is the skills of a real application at age 63). Every intermediate point is a real person from the sample.

Let us compare which areas the FACE path and the Wachter jump pass through. For each segment we take 50 points and look at the lowest KDE density on it; to make the number readable, we convert it into the share of real applications whose density is even lower.

In [ ]:
def min_density_share(points):
    ts = np.linspace(0, 1, 50)[:, None]
    low = min(kde((a + ts * (b - a)).T).min() for a, b in zip(points[:-1], points[1:]))
    return (dens_all < low).mean()

face_pts = pts[path]
jump_pts = np.vstack([tamara[1:], x_wachter[1:]])
for name, P in [("FACE", face_pts), ("Wachter", jump_pts)]:
    print("%-8s length in skills = %.2f, straight to the end = %.2f, "
          "share of applications with density below the sparsest point of the path = %.3f"
          % (name, sum(np.linalg.norm(b - a) for a, b in zip(P[:-1], P[1:])),
             np.linalg.norm(P[-1] - P[0]), min_density_share(P)))

In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(X[:, 1], X[:, 2], c=proba, cmap="Blues", s=6, vmin=0, vmax=1)
plt.colorbar(sc, label="p(invited) at age 63")
plt.plot(face_pts[:, 1], face_pts[:, 2], "-o", color="#eb6834", lw=2, ms=8, label="FACE path")
plt.plot(jump_pts[:, 1], jump_pts[:, 2], "--s", color="#1baf7a", lw=2, ms=8, label="Wachter jump")
plt.scatter([tamara[2]], [tamara[3]], color="black", s=60, zorder=5, label="Tamara")
plt.xlim(1.5, 6.5); plt.ylim(0, 4)
plt.xlabel("projects"); plt.ylabel("experience, years")
plt.title("Applications, FACE path and Wachter jump (projected on projects and experience)")
plt.legend(loc="upper right"); plt.show()

In the figure the dots are real applications projected on projects and experience, and the colour is the invitation probability at age 63. The orange polyline is the FACE path through real applications, the green dashed line is the Wachter jump straight to the boundary. Here both routes lie in a populated region: the sparsest point of the FACE path is denser than 66% of real applications, that of the Wachter jump denser than 55%. Tamara sits in the dense middle of the data, and there is no gap between her and the invited candidates; the FACE path goes through slightly denser places. FACE pays for a real endpoint and real intermediate steps with proximity: its endpoint is 1.93 away in skills along a straight line against 1.78 for Wachter, and its proximity is 2.07 against 1.69. If there were no dense trail, Dijkstra would find no path to the target applications — this is the FACE answer "no recourse".

One more detail of the FACE endpoint. The target was selected by the condition $p \ge 0.6$, but that $p$ comes from a forest that saw this application during training. Let us retrain the model on the same data without this single application and ask again.

In [ ]:
model_wo = make_model(df.drop(index=best - 1))     # node best is row best - 1 of df
print("p of the lesson model = %.3f | p of the model without this application = %.3f"
      % (p(x_face), p(x_face, model_wo)))

Without its own application the forest gives the FACE endpoint $p = 0.35$, below the threshold. A random forest is overconfident on its training objects: each application landed in the bootstrap samples of most trees. So the confidence threshold $t_p$ of FACE should be checked with predictions made without that object (out-of-fold) — otherwise the "real invited person" turns out to be invited only by the model that memorized them. Section 7 will show this once more.

## 6. Action costs in the spirit of Ustun

Distance in feature space is geometry, while cost is Tamara's effort (the step "Actionable Recourse" of lesson 4). Let us set rough prices in months: a course is 1 month, a project is 2 months, a year of experience is 12 months (it has to be lived through). The set of permitted actions $A(x)$ is integer: from 0 to 6 new courses, from 0 to 6 new projects and from 0 to 10 half-years of experience; skills cannot decrease and age cannot be touched. That is 539 actions, and an exhaustive enumeration over them replaces the integer programming of the `actionable-recourse` package: on a finite set it finds the exact minimum and proves just as exactly that no recourse exists.

**Task 4.** For every action in `acts`, compute its cost in months (`act_cost`) and find the cheapest valid action.

*Self-check question:* which plan is the cheapest, how many months does it cost and what is its $p$? How many months would the Wachter plan cost at the same prices?

<details><summary>Check yourself</summary>

The cheapest plan is four new courses (courses $2 \to 6$, projects and experience unchanged): 4 months, $p = 0.59$. The Wachter plan costs 16 months at the same prices (1.4 projects at 2 months each and 1.1 years of experience at 12 months each). Solution: `act_cost = acts @ np.array([1, 2, 6])` (a half-year of experience costs 6 months), `k = np.argmin(np.where(PA >= 0.5, act_cost, np.inf))`.
</details>

In [ ]:
acts = np.array(list(itertools.product(range(7), range(7), range(11))))   # +courses, +projects, +half-years of experience
XA = np.tile(tamara, (len(acts), 1))
XA[:, 1:] += acts * np.array([1., 1., 0.5])
PA = p_many(XA)
act_cost = ...  # Your code here: cost of every action in months
k = ...         # Your code here: index of the cheapest valid action
x_ustun = XA[k]
found["Ustun"] = x_ustun
print("cheapest plan:", acts[k], "→", x_ustun, "| cost =", act_cost[k],
      "months | p =", round(PA[k], 3))
print("cost of the Wachter plan at the same prices: %.1f months" % cost(x_wachter))

order = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))[:6]
pd.DataFrame({"+courses": acts[order, 0], "+projects": acts[order, 1], "+exp": acts[order, 2] * 0.5,
              "months": act_cost[order], "p": PA[order].round(3)})

The cheapest plan looks nothing like the closest one. Wachter moved projects and experience: the model is more sensitive to them (in the score their weights are 0.7 and 0.6 against 0.5 for courses), and per unit of distance they raise $p$ more. In effort, however, courses are the cheapest. Proximity and actionability are different quantities, and the price, not the geometry, decides between them.

Now let us narrow $A(x)$ to the real constraints of the next half-year: Tamara can manage at most two courses and two projects, and experience does not grow without a job.

In [ ]:
allowed = (acts[:, 0] <= 2) & (acts[:, 1] <= 2) & (acts[:, 2] == 0)
print("permitted actions:", allowed.sum(), "| valid among them:", (PA[allowed] >= 0.5).sum(),
      "| maximal p =", round(PA[allowed].max(), 3))

None of the 9 permitted actions gives $p \ge 0.5$: under these constraints no recourse exists for Tamara, and the enumeration proves it instead of assuming it. This is the kind of answer for which Ustun and co-authors formulated the task through a set of actions: "nothing available within half a year will help, experience is needed".

## 7. Robustness in the spirit of ROAR and PRR

Recourse is a promise over time (lesson 5). Let us check every plan from two sides:

- **model shift (ROAR):** the company retrains the screening model on new data. Like the function `make(seed)` from the step "Robust recourse for Tamara", we generate data with seeds 1–15 and train 15 models; the share of models with $p \ge 0.5$ is the robustness to the shift;
- **execution noise (PRR):** Tamara carries out the plan imprecisely. We add Gaussian noise to the skills with standard deviations of 0.5 courses, 0.5 projects and 0.7 years of experience (as in the lesson), 500 times; the share of valid executions is the robustness to noise.

Next to them we put the margin $p - 0.5$ and the proximity.

In [ ]:
t0 = time.time()
models = [make_model(make_data(s)) for s in range(1, 16)]
print("15 models in %.0f s" % (time.time() - t0))

**Task 5.** Complete `share_models(x)` — the share of the 15 retrained models in `models` with $p(x) \ge 0.5$. The noise function `share_noise(x)` is already written: it uses the same seed every time, so all plans are checked on the same perturbations.

*Self-check question:* what share of the retrained models invites Tamara under the Wachter plan and under the lesson 5 recipe "with a buffer" (courses 3, projects 4, experience 6.5)? What are their margins?

<details><summary>Check yourself</summary>

The Wachter plan is invited by 0 of the 15 retrained models (share 0.0), its margin is 0.005. The "buffer" plan is invited by 15 of 15 (share 1.0), its margin is 0.475. Solution: `return np.mean([p(x, mm) >= 0.5 for mm in models])`.
</details>

In [ ]:
def share_models(x):
    # Your code here
    ...

def share_noise(x, n=500):
    g = np.random.default_rng(3)                # execution noise on skills
    V = np.array(x, float) + g.normal(0, [0, 0.5, 0.5, 0.7], size=(n, 4))
    V[:, 1:] = np.clip(V[:, 1:], 0, None)
    return (p_many(V) >= 0.5).mean()

found["lesson 5: at boundary"] = np.array([63., 3., 4., 3.5])
found["lesson 5: buffer"] = np.array([63., 3., 4., 6.5])

true_thr = np.median(0.5*df.courses + 0.7*df.projects + 0.6*df.exp)   # score threshold without noise
def score_margin(x): return 0.5*x[1] + 0.7*x[2] + 0.6*x[3] - true_thr

rob = pd.DataFrame({name: {"p": round(p(x), 3), "margin": round(p(x) - 0.5, 3), "proximity": round(proximity(x), 2),
                           "score margin": round(score_margin(x), 2),
                           "share models": round(share_models(x), 3), "share noise": round(share_noise(x), 3)}
                    for name, x in found.items()}).T
rob

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, col, title in [(ax[0], "share models", "model shift (15 retrained)"),
                      (ax[1], "share noise", "execution noise (500 executions)")]:
    a.scatter(rob["margin"], rob[col], s=60, color="#2a78d6")
    for name, r in rob.iterrows():
        a.annotate(name, (r["margin"], r[col]), fontsize=8, xytext=(4, 3), textcoords="offset points")
    a.set_xlabel("margin = p − 0.5 under the current model"); a.set_ylabel("share valid")
    a.set_title(title); a.set_ylim(-0.05, 1.08); a.set_xlim(-0.03, 0.62)
plt.tight_layout(); plt.show()

In the two plots each dot is the plan of one method: horizontally its margin over the threshold under the current model, vertically the share of retrained models (left) and of noisy executions (right) under which the plan stays valid. The Wachter plan with a margin of 0.005 survives on none of the 15 retrained models and in only 19% of noisy executions; the "buffer" plan with a margin of 0.475 survives everywhere, but its proximity is 4.99 against 1.69 for Wachter.

The link between margin and robustness is rough, though: FACE has a margin of 0.125, yet the retrained models never invite it (its endpoint comes from the training sample, section 5), and the prototype plan at $\theta = 10$ has a margin of 0 and a share of models of 0.533. Our data is synthetic and we know the rule that labelled it, so the table has a "score margin" column — $0.5\cdot\text{courses} + 0.7\cdot\text{projects} + 0.6\cdot\text{experience}$ minus the median of this sum over the data. The shares of retrained models follow it almost exactly: plans with a negative score margin rely on a noise "pocket" of the current forest and disappear after retraining. A real task has no such rule, which is why ROAR and PRR check the perturbations themselves rather than the margin of a single model.

The last step is to build the robustness requirement into the search itself. On the integer actions from section 6 we find the cheapest plan that is valid **on all** 15 retrained models and on the current one (ROAR: the worst model of the set), and the cheapest plan that survives in at least 95% of noisy executions (PRR with $\delta = 0.05$).

In [ ]:
t0 = time.time()
worst = np.min([p_many(XA, mm) for mm in models + [model]], axis=0)     # worst model for every action
k_roar = np.argmin(np.where(worst >= 0.5, act_cost, np.inf))
cheap = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))
k_prr = next(k for k in cheap if share_noise(XA[k]) >= 0.95)
found["ROAR (worst of 16)"] = XA[k_roar]
found["PRR (δ=0.05)"] = XA[k_prr]
for name, k in [("Ustun", np.argmin(np.where(PA >= 0.5, act_cost, np.inf))), ("ROAR", k_roar), ("PRR", k_prr)]:
    print("%-6s +%s → %s | cost = %d months | p = %.3f" % (name, acts[k], XA[k], act_cost[k], PA[k]))
print("%.1f s" % (time.time() - t0))

## 8. Summary table of properties

Let us put all the plans into one table. The arrow in a header shows which way is better: ↑ larger, ↓ smaller. The cost in months is computed only for actionable plans (age untouched, no skill decreased).

In [ ]:
summary = pd.DataFrame({name: {
    "courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
    "validity ↑": p(x) >= 0.5,
    "proximity ↓": round(proximity(x), 2),
    "sparsity ↓": sparsity(x),
    "plausibility ↓": round(plausibility(x), 3),
    "actionable ↑": actionable(x),
    "months ↓": round(cost(x), 1) if actionable(x) else np.nan,
    "share models ↑": round(share_models(x), 3),
    "share noise ↑": round(share_noise(x), 3)} for name, x in found.items()}).T
summary

The table confirms the main conclusion of the lesson "Metrics and the main empirical fact": no method is the best in all columns at once.
- Wachter has the smallest proximity (1.69), but none of the 15 retrained models invites it, and under execution noise it is valid in 18.6% of cases.
- The prototype plan at $\theta = 10$ and FACE have the best plausibility (distances 0.125 and 0.129); yet no retrained model invites FACE, and the prototype plan costs 27.9 months.
- DiCE with the corridor and Ustun change the fewest skills (one each), but the first costs 40.8 months, and the second, the cheapest one (4 months), holds on only 1 of the 15 retrained models.
- DiCE-3 is valid but not actionable: it decreases the number of projects.
- Full robustness to model shift comes from the "buffer" plan (75 months) and from ROAR (8 months); ROAR, however, stays valid in a share of 0.668 of noisy executions, while PRR (also 8 months) is, conversely, valid in 99% of executions and on 11 of the 15 models.

The choice for Tamara depends on what matters more to her: the fastest is four courses from the Ustun plan, at the risk that they will not suffice after the screening model is retrained; the most reliable is the ROAR plan, two courses and three projects in 8 months, which withstands all 16 models.

## **Conclusions**

In this practice we answered Tamara with every method of the module on the same data and the same model and saw that:
- the counterfactual with the smallest distance (Wachter) sits on the decision boundary and is fragile under model retraining and imprecise execution;
- validity has to be checked with the model's prediction: DiCE cast the plans to the integer dtypes of the query and printed `invited = 1` for invalid points;
- DiCE gives a menu of plans, but `features_to_vary` does not forbid decreasing skills; the `permitted_range` corridor makes the plans actionable and at the same time uniform;
- the prototype sets a direction towards typical invited candidates but may itself be invalid; pulling towards it raises plausibility at the cost of proximity;
- FACE leads through real applications and shows a step-by-step path, and pays for this with proximity; the confidence threshold for its target should be checked with a model trained without that application;
- action costs change the answer more than the distance metric does, and an exhaustive enumeration over the set of actions can prove that no recourse exists;
- a margin over the threshold buys robustness to model shift and execution noise, and pays for it with proximity and months;
- no method is the best on all properties at once: the choice depends on which property is more expensive to get wrong in your task.